# Project Data Prep Notebook
**Data cleaning steps performed**
- Missing data handling
- Outlier handling
- Error correction
- Duplicate removal


**Feature engineering decisions**
- Single variable transforms
- New feature definitions
- Categorical encoding decisions
- Normalization/standardization decisions
- Date/time transformations 


**Feature selection decisions**
- Variable to remove and rationale


**Dataset partitioning decisions (oversampling/undersampling)**


**Final dataset summary**
- Number of rows/columns
- Summary of key variables
- Summary of transformations and cleaning steps performed


In [3]:
#imports 
import pandas as pd
import numpy as np
import scipy.stats as stats

In [4]:
df_raw = pd.read_csv(
    "../data/interim/healthcare_readmissions_dataset_train_post_eda.csv",
    keep_default_na=False,
    na_values=[""],
)

In [5]:
""" 
loading EDA dataset; has age outliers removed but thats it 
"""

df_raw.head()

,patient_id,age,gender,ethnicity,hospital_id,height_m,smoker,bmi,weight_kg,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,medications_prescribed,length_of_stay,type_of_treatment,readmission_within_30_days
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,0,0,Regular,High-fat,3.0,3.0,0,None,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,0,0,Regular,High-fat,2.0,NaN,2,None,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,0,1,None,Other,NaN,2.0,5,None,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,0,0,None,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,0,0,Occasional,High-fat,7.0,5.0,7,Major Surgery,1


## **DATA CLEANING**

### Missing data handling

In [6]:
df_raw.info() # only missings in number_of_prior_visits and medications_prescribed

<class 'pandas.DataFrame'>
RangeIndex: 7958 entries, 0 to 7957
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   patient_id                  7958 non-null   int64  
 1   age                         7958 non-null   int64  
 2   gender                      7958 non-null   str    
 3   ethnicity                   7958 non-null   str    
 4   hospital_id                 7958 non-null   str    
 5   height_m                    7958 non-null   float64
 6   smoker                      7958 non-null   bool   
 7   bmi                         7958 non-null   float64
 8   weight_kg                   7958 non-null   float64
 9   adjusted_weight_kg          7958 non-null   float64
 10  has_diabetes                7958 non-null   int64  
 11  has_hypertension            7958 non-null   int64  
 12  exercise_frequency          7958 non-null   str    
 13  diet_type                   7958 non-null   

In [7]:
# check for overlaps in missings

overlap = df_raw['number_of_prior_visits'].isna() & df_raw['medications_prescribed'].isna()
both_missing = df_raw[df_raw['number_of_prior_visits'].isna() & df_raw['medications_prescribed'].isna()]

print(f"Number of rows with both 'number_of_prior_visits' and 'medications_prescribed' missing: {both_missing.shape[0]}")




Number of rows with both 'number_of_prior_visits' and 'medications_prescribed' missing: 21


Not a lot of overlap in missingness, which means removing all the rows would probably not work.

Will return to imputing later as columns will be changed potentially. 

### Outlier handling

In [8]:
# looking into potential bmi outliers

threshold = 3.5
z_scores = np.abs(stats.zscore(df_raw['bmi']))
outliers = df_raw[z_scores > threshold]
print(f"Identified {len(outliers)} BMI outliers at threshold {threshold}:")

display(outliers)

Identified 7 BMI outliers at threshold 3.5:


,patient_id,age,gender,ethnicity,hospital_id,height_m,smoker,bmi,weight_kg,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,medications_prescribed,length_of_stay,type_of_treatment,readmission_within_30_days
792,1000997,55,Female,African American,Hosp2,1.7,False,43.0,124.3,123.805452,0,0,Regular,Vegetarian,2.0,1.0,2,None,0
797,1001005,36,Male,African American,Hosp1,1.7,True,43.7,126.3,125.833624,0,1,Occasional,High-fat,2.0,0.0,0,Major Surgery,0
3878,1004893,43,Male,African American,Hosp1,1.6,False,44.0,112.6,112.287576,1,0,None,High-fat,1.0,3.0,1,Minor Surgery,0
4444,1005602,63,Female,Hispanic,Hosp1,1.7,True,9.3,26.9,26.764915,1,0,Regular,Balanced,2.0,2.0,1,None,0
4771,1006027,18,Female,Hispanic,Hosp1,1.7,False,43.5,125.7,125.390224,1,1,Occasional,Balanced,2.0,3.0,3,Minor Surgery,0
4820,1006086,51,Male,Hispanic,Hosp3,1.9,False,43.8,158.1,157.360966,0,0,None,High-fat,NaN,6.0,4,None,0
6398,1008069,67,Female,African American,Hosp1,1.7,True,8.3,24.0,23.891220,0,0,None,Balanced,3.0,2.0,0,None,0


I think these rows are safe to delete, the bmis are just too extreme to be worth considering. 

In [9]:
df_transformed = df_raw.drop(outliers.index)

In [10]:
df_transformed.info() # now at 7951 x 19 columns

<class 'pandas.DataFrame'>
Index: 7951 entries, 0 to 7957
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   patient_id                  7951 non-null   int64  
 1   age                         7951 non-null   int64  
 2   gender                      7951 non-null   str    
 3   ethnicity                   7951 non-null   str    
 4   hospital_id                 7951 non-null   str    
 5   height_m                    7951 non-null   float64
 6   smoker                      7951 non-null   bool   
 7   bmi                         7951 non-null   float64
 8   weight_kg                   7951 non-null   float64
 9   adjusted_weight_kg          7951 non-null   float64
 10  has_diabetes                7951 non-null   int64  
 11  has_hypertension            7951 non-null   int64  
 12  exercise_frequency          7951 non-null   str    
 13  diet_type                   7951 non-null   str  

EDA determined there were no duplicates, and also got rid of the obvious "errors" (impossible ages). 

## **FEATURE ENGINEERING**

### Single Variable Transforms

In [11]:
# turning medications_prescribed to binary

df_transformed = df_transformed.copy() # to avoid SettingWithCopyWarning

df_transformed["medications_prescribed"] = df_transformed["medications_prescribed"].replace("", pd.NA).astype(float)
df_transformed["is_prescribed"] = df_transformed["medications_prescribed"].apply(lambda x: 1 if x > 0 else 0)
# imputing to false/0 by default for now; will be investigated later

In [12]:
df_transformed.drop(columns=["medications_prescribed"], inplace=True)

In [13]:
df_transformed.info()

<class 'pandas.DataFrame'>
Index: 7951 entries, 0 to 7957
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   patient_id                  7951 non-null   int64  
 1   age                         7951 non-null   int64  
 2   gender                      7951 non-null   str    
 3   ethnicity                   7951 non-null   str    
 4   hospital_id                 7951 non-null   str    
 5   height_m                    7951 non-null   float64
 6   smoker                      7951 non-null   bool   
 7   bmi                         7951 non-null   float64
 8   weight_kg                   7951 non-null   float64
 9   adjusted_weight_kg          7951 non-null   float64
 10  has_diabetes                7951 non-null   int64  
 11  has_hypertension            7951 non-null   int64  
 12  exercise_frequency          7951 non-null   str    
 13  diet_type                   7951 non-null   str  

In [14]:
# converting to numeric
df_transformed["number_of_prior_visits"] = df_transformed["number_of_prior_visits"].replace("", pd.NA).astype(float)

In [15]:
""" 
converting Length of Stay to a length of stay score.
Encoded the same way as used in the LACE Index (Length of stay, Acuity of admission, Comorbidity, Emergency department use),
a popular readmission risk scoring system.
"""
def add_length_of_stay_score(df: pd.DataFrame) -> pd.DataFrame:
    """Converts length_of_stay (days) to an ordinal risk score (1–7)."""
    df = df.copy()
    def _score(x):
        if x <= 1: return 1
        if x <= 2: return 2
        if x <= 3: return 3
        if x <= 6: return 4
        if x <= 14: return 5
        return 7
    df["length_of_stay_score"] = df["length_of_stay"].apply(_score)
    return df


In [16]:
df_transformed = add_length_of_stay_score(df_transformed)

In [17]:
df_transformed.drop(columns=["length_of_stay"], inplace=True) # length_of_stay replaced by length_of_stay_score

In [18]:
df_transformed.info() # bmi outliers removed, missings still the same

<class 'pandas.DataFrame'>
Index: 7951 entries, 0 to 7957
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   patient_id                  7951 non-null   int64  
 1   age                         7951 non-null   int64  
 2   gender                      7951 non-null   str    
 3   ethnicity                   7951 non-null   str    
 4   hospital_id                 7951 non-null   str    
 5   height_m                    7951 non-null   float64
 6   smoker                      7951 non-null   bool   
 7   bmi                         7951 non-null   float64
 8   weight_kg                   7951 non-null   float64
 9   adjusted_weight_kg          7951 non-null   float64
 10  has_diabetes                7951 non-null   int64  
 11  has_hypertension            7951 non-null   int64  
 12  exercise_frequency          7951 non-null   str    
 13  diet_type                   7951 non-null   str  

In [19]:
"""
binning age feature
Bins: 0–18, 19–25, 26–40, 41–65, 66–80, 81+
Rationale: age has a non-linear relationship with readmission risk.
Binning captures clinical risk tiers more effectively than raw continuous values
Raw age column dropped after binning
"""

age_bins = [0, 18, 25, 40, 65, 80, np.inf]
age_labels = ["0-18", "19-25", "26-40", "41-65", "66-80", "81+"]
df_transformed["age_group"] = pd.cut(df_transformed["age"], bins=age_bins, labels=age_labels, right=False)
df_transformed.drop(columns=["age"], inplace=True) # age replaced by age_group

In [20]:
# dropping weight as adjusted_weight is hghly correlated and ecists for a reason (i asuszme)
df_transformed.drop(columns=["weight_kg"], inplace=True) 

In [21]:
display(df_transformed.head())

,patient_id,gender,ethnicity,hospital_id,height_m,smoker,bmi,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,type_of_treatment,readmission_within_30_days,is_prescribed,length_of_stay_score,age_group
0,1000000,Female,African American,Hosp2,1.6,False,25.0,63.283346,0,0,Regular,High-fat,3.0,None,0,1,1,19-25
1,1000002,Female,Hispanic,Hosp3,1.8,True,27.0,87.678859,0,0,Regular,High-fat,2.0,None,0,0,2,41-65
2,1000003,Male,African American,Hosp1,1.8,False,35.0,113.497844,0,1,None,Other,NaN,None,0,1,4,26-40
3,1000004,Female,Caucasian,Hosp2,1.8,False,27.7,89.717694,0,0,None,Other,3.0,Major Surgery,0,0,1,66-80
4,1000005,Female,Hispanic,Hosp1,1.9,False,22.4,80.528927,0,0,Occasional,High-fat,7.0,Major Surgery,1,1,5,41-65


Notes for future investigation:
- number of prior visits could potentially be imputed based on type of treatment (?) look into perhaps


### Feature Selection Decisions

- dropped weight_kg as adjusted_weight_kg is hghly correlated and I assume would not be added to the dataset for no reason.
- age and length_of_stay were transformed into scores, so the original variables were dropped.
- non-categorical data will be standardized but will be done later
- SMOTE and other oversmapling techniques will be tested in experimenting phase

## **DATASET SUMMARY**

In [22]:
display(df_transformed.head())

,patient_id,gender,ethnicity,hospital_id,height_m,smoker,bmi,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,type_of_treatment,readmission_within_30_days,is_prescribed,length_of_stay_score,age_group
0,1000000,Female,African American,Hosp2,1.6,False,25.0,63.283346,0,0,Regular,High-fat,3.0,None,0,1,1,19-25
1,1000002,Female,Hispanic,Hosp3,1.8,True,27.0,87.678859,0,0,Regular,High-fat,2.0,None,0,0,2,41-65
2,1000003,Male,African American,Hosp1,1.8,False,35.0,113.497844,0,1,None,Other,NaN,None,0,1,4,26-40
3,1000004,Female,Caucasian,Hosp2,1.8,False,27.7,89.717694,0,0,None,Other,3.0,Major Surgery,0,0,1,66-80
4,1000005,Female,Hispanic,Hosp1,1.9,False,22.4,80.528927,0,0,Occasional,High-fat,7.0,Major Surgery,1,1,5,41-65


In [23]:
display(df_transformed.info())

<class 'pandas.DataFrame'>
Index: 7951 entries, 0 to 7957
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   patient_id                  7951 non-null   int64   
 1   gender                      7951 non-null   str     
 2   ethnicity                   7951 non-null   str     
 3   hospital_id                 7951 non-null   str     
 4   height_m                    7951 non-null   float64 
 5   smoker                      7951 non-null   bool    
 6   bmi                         7951 non-null   float64 
 7   adjusted_weight_kg          7951 non-null   float64 
 8   has_diabetes                7951 non-null   int64   
 9   has_hypertension            7951 non-null   int64   
 10  exercise_frequency          7951 non-null   str     
 11  diet_type                   7951 non-null   str     
 12  number_of_prior_visits      7640 non-null   float64 
 13  type_of_treatment           7951 n

None

In [24]:
# imputing number_of_prior_visits

df_transformed["number_of_prior_visits"] = df_transformed["number_of_prior_visits"].fillna(0)

In [25]:
df_transformed.info()

<class 'pandas.DataFrame'>
Index: 7951 entries, 0 to 7957
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   patient_id                  7951 non-null   int64   
 1   gender                      7951 non-null   str     
 2   ethnicity                   7951 non-null   str     
 3   hospital_id                 7951 non-null   str     
 4   height_m                    7951 non-null   float64 
 5   smoker                      7951 non-null   bool    
 6   bmi                         7951 non-null   float64 
 7   adjusted_weight_kg          7951 non-null   float64 
 8   has_diabetes                7951 non-null   int64   
 9   has_hypertension            7951 non-null   int64   
 10  exercise_frequency          7951 non-null   str     
 11  diet_type                   7951 non-null   str     
 12  number_of_prior_visits      7951 non-null   float64 
 13  type_of_treatment           7951 n

In [26]:
df_final = df_transformed.copy()

In [33]:
info_df = pd.DataFrame({
    "Column": df_final.columns,
    "Non-Null Count": df_final.count().values,
    "Dtype": df_final.dtypes.astype(str).values,
})
print(info_df.to_latex(index=False, caption="Dataset Info", label="tab:dataset_info"))

\begin{table}
\caption{Dataset Info}
\label{tab:dataset_info}
\begin{tabular}{lrl}
\toprule
Column & Non-Null Count & Dtype \\
\midrule
patient_id & 7951 & int64 \\
gender & 7951 & str \\
ethnicity & 7951 & str \\
hospital_id & 7951 & str \\
height_m & 7951 & float64 \\
smoker & 7951 & bool \\
bmi & 7951 & float64 \\
adjusted_weight_kg & 7951 & float64 \\
has_diabetes & 7951 & int64 \\
has_hypertension & 7951 & int64 \\
exercise_frequency & 7951 & str \\
diet_type & 7951 & str \\
number_of_prior_visits & 7951 & float64 \\
type_of_treatment & 7951 & str \\
readmission_within_30_days & 7951 & int64 \\
is_prescribed & 7951 & int64 \\
length_of_stay_score & 7951 & int64 \\
age_group & 7951 & category \\
\bottomrule
\end{tabular}
\end{table}



In [ ]:
\begin{table}
\caption{Dataset Info}
\label{tab:dataset_info}
\begin{tabular}{lrl}
\toprule
Column & Non-Null Count & Dtype \\
\midrule
patient_id & 7951 & int64 \\
gender & 7951 & str \\
ethnicity & 7951 & str \\
hospital_id & 7951 & str \\
height_m & 7951 & float64 \\
smoker & 7951 & bool \\
bmi & 7951 & float64 \\
adjusted_weight_kg & 7951 & float64 \\
has_diabetes & 7951 & int64 \\
has_hypertension & 7951 & int64 \\
exercise_frequency & 7951 & str \\
diet_type & 7951 & str \\
number_of_prior_visits & 7951 & float64 \\
type_of_treatment & 7951 & str \\
readmission_within_30_days & 7951 & int64 \\
is_prescribed & 7951 & int64 \\
length_of_stay_score & 7951 & int64 \\
age_group & 7951 & category \\
\bottomrule
\end{tabular}
\end{table}

In [ ]:
df_final.to_csv("../data/interim/healthcare_readmissions_dataset_train_preprocessed.csv", index=False)